# 03 — Associations, under-studied signals, contradictions, nulls

The four discovery tables: how each was filtered, how to reproduce the headline counts, and what
each one cannot tell you.

> ## Scope — read this first
>
> **This is Tier 1: POTS / dysautonomia literature ONLY** — chunks 0–75 of a 286-chunk corpus,
> **~3,121 of ~14,263 corpus papers**.
>
> ME/CFS, fibromyalgia, EDS/hypermobility, MCAS, IBS, migraine and autism appear here **only as
> incidental mentions inside POTS/dysautonomia papers**. Those conditions' own literatures are in
> Tiers 2–4 and are **not yet extracted**. This is **not** a cross-condition resource.
>
> **Not medical advice.** Machine-extracted research information, not clinical guidance. No
> diagnostic or treatment recommendations. Personal health questions belong with a qualified
> clinician.
>
> **Attribution.** Every finding links to its source paper with authors and journal. These are
> automated summaries of other researchers' work — the credit is theirs, and the original paper is
> the authority. `provenance_quote_trunc` is short verbatim publisher-copyright text included for
> verification only; always keep it next to its citation and never bulk-export it.

In [1]:
# ── Data location ──────────────────────────────────────────────────────────────
# Repoint this constant at your own fork if you want. Raw GitHub is used in Colab;
# a local ./data or ../data directory (i.e. running inside a clone) wins if present.
RAW_BASE = "https://raw.githubusercontent.com/USER/REPO/main/data"

import os, io, json, urllib.request
import pandas as pd, numpy as np
import matplotlib
import matplotlib.pyplot as plt

_LOCAL = next((p for p in ("data", "../data", "../../data") if os.path.isdir(p)), None)
_CACHE = ".cache_tier1"; os.makedirs(_CACHE, exist_ok=True)

def data_path(name):
    """Return a readable path for a data file: local clone if present, else download once."""
    if _LOCAL and os.path.exists(os.path.join(_LOCAL, name)):
        return os.path.join(_LOCAL, name)
    dst = os.path.join(_CACHE, name)
    if not os.path.exists(dst):
        url = f"{RAW_BASE}/{name}"
        print(f"downloading {name} ...")
        urllib.request.urlretrieve(url, dst)
    return dst

def load(name):
    p = data_path(name)
    if p.endswith(".parquet"): return pd.read_parquet(p)
    if p.endswith(".json"):    return json.load(open(p))
    return pd.read_csv(p, low_memory=False)

print("data source:", _LOCAL if _LOCAL else RAW_BASE)

data source: ../data


In [2]:
F  = load("tier1_findings.parquet")
A  = load("association_table.parquet")
U  = load("understudied_signals.parquet")
C  = load("contradictions.parquet")
NL = load("null_results.parquet")
CV = load("convergence_tier1.parquet")
plt.rcParams.update({"font.size": 9, "axes.spines.top": False, "axes.spines.right": False, "figure.dpi": 110})
print({k: v.shape for k, v in dict(assoc=A, understudied=U, contradictions=C, nulls=NL, convergence=CV).items()})

{'assoc': (555, 29), 'understudied': (329, 31), 'contradictions': (532, 42), 'nulls': (5324, 43), 'convergence': (308, 31)}


## 1. Associations

Entity pairs co-mentioned in the same paper, tested against a hypergeometric null with
Benjamini-Hochberg FDR correction over the 547 non-trivial pairs.

> **Co-occurrence is co-mention, not interaction.** The strongest pairs include the standard POTS
> drug ladder, which co-occurs because management reviews list those drugs together. Significance
> there validates the pipeline; it is not a discovery.

In [3]:
print("total pairs                      :", len(A))
print("tested (included_in_primary)     :", int(A.included_in_primary.sum()))
print("significant at BH q<0.05         :", int(A.loc[A.included_in_primary, "significant_fdr05"].sum()))
print("paper universe                   :", int(A.n_papers_universe.max()))
print()
print("mechanical-explanation flags (why a pair may be trivial):")
for c in ["trivial_component_link","name_level_nested","surface_string_linked"]:
    print(f"  {c:24s} {int(A[c].sum()):>4}")

total pairs                      : 555
tested (included_in_primary)     : 547
significant at BH q<0.05         : 456
paper universe                   : 2176

mechanical-explanation flags (why a pair may be trivial):
  trivial_component_link      8
  name_level_nested           7
  surface_string_linked     107


### The 101 cross-kind, non-trivial pairs

Cross-kind means the two entities are different *types* (a drug and an analyte, a gene and a
physiological measure) — where a co-mention is less likely to be pure reporting convention. The
filter is: significant, in the primary tested set, different `entity_kind`, and **not** flagged
`trivial_component_link`, `name_level_nested`, or `surface_string_linked`.

In [4]:
cross = A[A.included_in_primary & A.significant_fdr05 & (A.entity_kind_a != A.entity_kind_b)
          & ~A.trivial_component_link & ~A.name_level_nested & ~A.surface_string_linked]
print("cross-kind, non-trivial, significant:", len(cross))
print()
print("CAVEAT: map_confidence is NOT filtered here by design - both sides are exposed as columns.")
print("  both sides high confidence:", int(((cross.map_confidence_a=='high') & (cross.map_confidence_b=='high')).sum()))
print()
cross.sort_values("lift", ascending=False).head(15)[
    ["entity_a","entity_b","entity_kind_a","entity_kind_b","n_papers_both","lift","q_bh","map_confidence_a","map_confidence_b"]]

cross-kind, non-trivial, significant: 101

CAVEAT: map_confidence is NOT filtered here by design - both sides are exposed as columns.
  both sides high confidence: 84



In [5]:
t = cross.sort_values("lift", ascending=False).head(15)[::-1]
lab = [f"{a} × {b}" for a, b in zip(t.entity_a, t.entity_b)]
fig, ax = plt.subplots(figsize=(8.6, 5.4))
ax.barh(range(len(t)), t.lift.values, color="#4C72B0")
for i, (l, n) in enumerate(zip(t.lift.values, t.n_papers_both.values)):
    ax.text(l * 1.03, i, f"lift {l:.0f} · {int(n)} papers", va="center", fontsize=7.5)
ax.set_yticks(range(len(t))); ax.set_yticklabels(lab, fontsize=8)
ax.set_xscale("log"); ax.set_xlim(t.lift.min()*0.7, t.lift.max()*3.2)
ax.set_xlabel("co-mention lift (log scale)")
ax.set_title("Cross-kind associations by lift — co-mention in one paper,\nNOT evidence of interaction", loc="left")
fig.tight_layout(); fig.savefig("fig_assoc_crosskind.png")
print(t[["entity_a","entity_b","lift","n_papers_both","q_bh"]].iloc[::-1].to_string(index=False))

                     entity_a                            entity_b    lift  n_papers_both         q_bh
                         ccl5                               il-21 544.000              3 2.278415e-08
                        hemin                                hmbs 408.000              3 7.288412e-08
                     atropine muscarinic m2 receptor autoantibody 272.000              3 2.930095e-07
                sphingolipids                              sptlc2 217.600              3 6.063196e-07
               erythropoietin                          octreotide 204.000              3 6.666149e-07
          1-deoxysphingolipid                              sptlc2 181.333              3 1.096510e-06
montreal cognitive assessment                           scopa-aut 161.185              4 2.047519e-08
                     l-serine                              sptlc1 145.067              4 1.541312e-08
   nmda receptor autoantibody                    ovarian teratoma 133.224         

## 2. Under-studied signals

Entity × condition × direction records with thin evidence — built as a **hypothesis queue for
human triage**, not a list of true-but-ignored facts.

> At 2–3 papers, a genuinely neglected finding and a single-lab artifact are statistically
> indistinguishable. There is no filter that keeps one and removes the other. Requiring ≥2
> independent last authors reduces single-lab risk; it does not eliminate it.

In [6]:
print("candidates                          :", len(U))
print("distinct entities                   :", U.canonical_entity.nunique())
print("is_thin (<=15 papers corpus-wide)   :", int(U.is_thin.sum()))
print("headline_eligible (thin & high conf):", int(U.headline_eligible.sum()))
rep = U[U.headline_eligible & (U.n_distinct_last_authors >= 2) & ~U.case_report_only]
print("+ >=2 indep. last authors, not case-report-only:", len(rep))
print()
print("map_confidence of candidates:"); print(U.map_confidence.value_counts())
print("\n21 thin candidates are dropped by the high-confidence filter:",
      int((U.is_thin & (U.map_confidence != 'high')).sum()))

candidates                          : 329
distinct entities                   : 262
is_thin (<=15 papers corpus-wide)   : 247
headline_eligible (thin & high conf): 226
+ >=2 indep. last authors, not case-report-only: 198

map_confidence of candidates:
map_confidence
high      300
medium     21
low         8
Name: count, dtype: int64

21 thin candidates are dropped by the high-confidence filter: 21


In [7]:
cols = ["canonical_entity","entity_kind","condition","direction","n_papers","n_distinct_last_authors",
        "direction_agreement","best_tier","mean_year","replication_score"]
print("Top 20 replication targets (headline-eligible, >=2 independent last authors, not case-report-only)")
rep.sort_values("replication_score", ascending=False).head(20)[cols]

Top 20 replication targets (headline-eligible, >=2 independent last authors, not case-report-only)


In [8]:
t = rep.sort_values("replication_score", ascending=False).head(18)[::-1]
fig, ax = plt.subplots(figsize=(8.8, 5.8))
sizes = 22 + 10 * t.n_papers.values
sc = ax.scatter(t.replication_score.values, range(len(t)), s=sizes,
                c=t.direction_agreement.values, cmap="viridis", vmin=0, vmax=1, edgecolor="#333333", lw=0.5)
for i, (s, npap, nla) in enumerate(zip(t.replication_score.values, t.n_papers.values, t.n_distinct_last_authors.values)):
    ax.text(s + 0.012, i, f"{int(npap)} papers · {int(nla)} labs", va="center", fontsize=7)
ax.set_yticks(range(len(t)))
ax.set_yticklabels([f"{e} ({c})" for e, c in zip(t.canonical_entity, t.condition)], fontsize=7.5)
ax.set_xlabel("replication score"); ax.set_xlim(t.replication_score.min()-0.03, t.replication_score.max()+0.10)
ax.set_title("Replication targets — marker size = papers, colour = direction agreement.\nThin evidence includes noise; every row needs a human.", loc="left")
fig.colorbar(sc, ax=ax, label="direction agreement", shrink=0.6)
fig.tight_layout(); fig.savefig("fig_understudied.png")
print(t[["canonical_entity","condition","n_papers","n_distinct_last_authors","direction_agreement","replication_score"]].iloc[::-1].to_string(index=False))

             canonical_entity              condition  n_papers  n_distinct_last_authors  direction_agreement  replication_score
          incobotulinumtoxina           dysautonomia         2                        2                  1.0             71.323
       trimethylamine n-oxide                general         2                        2                  1.0             67.361
  rem sleep behavior disorder           dysautonomia         6                        6                  1.0             60.637
                postural sway           dysautonomia         2                        2                  1.0             59.372
                      slc18a3           dysautonomia         2                        2                  1.0             59.372
                      syncope           dysautonomia         3                        3                  1.0             58.035
           hoehn & yahr stage           dysautonomia         2                        2                 

## 3. Contradictions

Direction flips and null-vs-positive conflicts. **The most important column is
`broad_condition_flag`** — 376 of 532 conflicts sit under an umbrella condition label and may
reflect pooled distinct diseases rather than a replication failure. Two studies disagreeing about
"dysautonomia" may both be right about different patient populations.

In [9]:
print("total conflicts               :", len(C))
print(C.conflict_type.value_counts().to_string())
print()
print("has_visible_moderator         :", int(C.has_visible_moderator.sum()))
print("broad_condition_flag          :", int(C.broad_condition_flag.sum()),
      f"({100*C.broad_condition_flag.mean():.1f}% - may be pooling, not replication failure)")
print("contradicts_ref_corroborated  :", int(C.contradicts_ref_corroborated.sum()))
print()
print("A same-paper filter removed 61 candidate hard flips and 97 null-vs-positive candidates")
print("where both sides came from ONE paper (internal hedging, not literature controversy).")

total conflicts               : 532
conflict_type
hard-direction-flip    268
null-vs-positive       264

has_visible_moderator         : 127
broad_condition_flag          : 376 (70.7% - may be pooling, not replication failure)
contradicts_ref_corroborated  : 224

A same-paper filter removed 61 candidate hard flips and 97 null-vs-positive candidates
where both sides came from ONE paper (internal hedging, not literature controversy).


In [10]:
mod = C[C.has_visible_moderator]
print(f"{len(mod)} conflicts have a visible moderator that plausibly explains the disagreement.\n")
from collections import Counter
cnt = Counter()
for v in mod.moderator_candidates.dropna():
    for part in str(v).replace(";", ",").split(","):
        if part.strip(): cnt[part.strip()] += 1
print("moderator candidate frequency:")
for k, n in cnt.most_common(12): print(f"  {k:32s} {n}")

127 conflicts have a visible moderator that plausibly explains the disagreement.

moderator candidate frequency:
  fulltext_status                  61
  population_age_group             36
  sex_hormonal                     21
  is_covid_associated              17
  pots_subtype                     4


In [11]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.9))
ct = C.conflict_type.value_counts()[::-1]
axes[0].barh(range(len(ct)), ct.values, color="#DD8452")
for i, n in enumerate(ct.values): axes[0].text(n + 4, i, f"{n:,}", va="center", fontsize=8)
axes[0].set_yticks(range(len(ct))); axes[0].set_yticklabels(ct.index, fontsize=8)
axes[0].set_xlim(0, ct.max()*1.2); axes[0].set_title("Conflict type", loc="left")
flags = pd.Series({"visible moderator": int(C.has_visible_moderator.sum()),
                   "broad condition label\n(may be pooling)": int(C.broad_condition_flag.sum()),
                   "ref corroborated": int(C.contradicts_ref_corroborated.sum())})[::-1]
axes[1].barh(range(len(flags)), flags.values, color=["#55A868","#C44E52","#4C72B0"][::-1])
for i, n in enumerate(flags.values): axes[1].text(n + 5, i, f"{n:,} / {len(C):,}", va="center", fontsize=8)
axes[1].set_yticks(range(len(flags))); axes[1].set_yticklabels(flags.index, fontsize=8)
axes[1].set_xlim(0, len(C)*1.15); axes[1].set_title("Interpretation flags — read these before\ncalling anything a replication failure", loc="left")
top_m = pd.Series(dict(cnt.most_common(8)))[::-1]
axes[2].barh(range(len(top_m)), top_m.values, color="#8172B3")
for i, n in enumerate(top_m.values): axes[2].text(n + 1, i, str(n), va="center", fontsize=8)
axes[2].set_yticks(range(len(top_m))); axes[2].set_yticklabels(top_m.index, fontsize=7.5)
axes[2].set_xlim(0, top_m.max()*1.22); axes[2].set_title("Moderator candidates", loc="left")
fig.tight_layout(); fig.savefig("fig_contradictions.png")

In [12]:
cols = ["canonical_entity","condition","conflict_type","side_A_direction","n_papers_A","side_B_direction",
        "n_papers_B","has_visible_moderator","broad_condition_flag","moderator_detail"]
print("Strongest conflicts WITH a visible moderator (the tractable ones):")
mod.sort_values("conflict_strength", ascending=False).head(12)[cols]

Strongest conflicts WITH a visible moderator (the tractable ones):


## 4. Null and failed results

The usually-invisible half of the literature. Extracted deliberately rather than dropped.

In [13]:
print(NL.null_class.value_counts().to_string())
print()
tf = NL[NL.null_class.str.contains("treatment")]
rct = tf[tf.evidence_tier.isin(["RCT","meta-analysis"])]
print(f"treatment failures            : {len(tf):,}")
print(f"  at RCT / meta-analysis tier : {len(rct)}   <- the most decision-relevant subset here")
print(f"distinct papers with a null   : {NL.canonical_pmid.nunique():,}")
print()
print("negative_strength:"); print(NL.negative_strength.value_counts().to_string())

null_class
biomarker/mechanism null          3076
treatment/intervention failure    1340
diagnostic null                    908

treatment failures            : 1,340
  at RCT / meta-analysis tier : 315   <- the most decision-relevant subset here
distinct papers with a null   : 1,996

negative_strength:
negative_strength
3.00    854
2.00    776
1.20    649
0.60    429
2.50    420
2.10    386
1.40    227
1.80    197
4.50    168
0.72    147
0.80    145
0.48    140
1.75    126
0.84    123
0.36    105
0.56     97
3.15     77
1.50     76
3.50     55
5.00     51
0.42     46
2.70     30


In [14]:
cols = ["canonical_entity","treatment_class","condition","evidence_tier","n_participants",
        "negative_strength","citation_short","pubmed_url"]
print(f"All {len(rct)} RCT/meta-tier treatment failures — first 20 shown")
print("(caveat: 'failed in this trial' is not 'ineffective'; check comparator, dose and population)")
rct[cols].head(20)

All 315 RCT/meta-tier treatment failures — first 20 shown
(caveat: 'failed in this trial' is not 'ineffective'; check comparator, dose and population)


In [15]:
byent = (rct.groupby("canonical_entity").agg(papers=("canonical_pmid","nunique"))
         .sort_values("papers", ascending=False).head(15)[::-1])
fig, ax = plt.subplots(figsize=(7.8, 4.6))
ax.barh(range(len(byent)), byent.papers.values, color="#C44E52")
for i, n in enumerate(byent.papers.values): ax.text(n + 0.05, i, str(int(n)), va="center", fontsize=8)
ax.set_yticks(range(len(byent))); ax.set_yticklabels(byent.index, fontsize=8)
ax.set_xlim(0, byent.papers.max()*1.2); ax.set_xlabel("distinct papers reporting a failure")
ax.set_title("Entities with RCT/meta-tier treatment failures\n'failed in this trial' is not 'ineffective' — check dose, comparator, population", loc="left")
fig.tight_layout(); fig.savefig("fig_null_rct.png")
print(byent[::-1].to_string())

                            papers
canonical_entity                  
midodrine                        6
vagus nerve stimulation          4
intravenous immunoglobulin       4
pregabalin                       3
ssri                             3
heart rate variability           3
atomoxetine                      3
fludrocortisone                  3
nirmatrelvir/ritonavir           3
pyridostigmine                   3
milnacipran                      2
glp-1 receptor agonist           2
pramipexole                      2
orthostatic hypotension          2
propranolol                      2


## 5. Convergence — and why it is gated hard

Entities pointing the same direction across multiple conditions. **In Tier 1 the non-POTS support
is incidental mention**, because Tier 1 contains only POTS/dysautonomia papers. Hence two gates
before any shared-mechanism language.

In [16]:
print("records                         :", len(CV))
print("passes_nonpots_bar (>=2 non-POTS):", int(CV.passes_nonpots_bar.sum()),
      "->", CV[CV.passes_nonpots_bar].canonical_entity.nunique(), "distinct entities")
print("headline_eligible (bar & uncontested):", int(CV.headline_eligible.sum()))
print("contested (>=2 opposite-direction papers):", int(CV.is_contested.sum()))
print()
print("NOTE: none of the highest-scoring entities are headline-eligible -")
print("heart rate, HRV, norepinephrine, RMSSD, SDNN, IL-6 all fail on contestation:")
CV.sort_values("score_multicondition", ascending=False).head(8)[
    ["canonical_entity","concordant_direction","n_conditions","n_papers","n_papers_other_condition",
     "is_contested","passes_nonpots_bar","headline_eligible"]]

records                         : 308
passes_nonpots_bar (>=2 non-POTS): 106 -> 92 distinct entities
headline_eligible (bar & uncontested): 45
contested (>=2 opposite-direction papers): 164

NOTE: none of the highest-scoring entities are headline-eligible -
heart rate, HRV, norepinephrine, RMSSD, SDNN, IL-6 all fail on contestation:


In [17]:
he = CV[CV.headline_eligible].sort_values("score_multicondition", ascending=False)
print(f"{len(he)} headline-eligible convergence records:")
he.head(20)[["canonical_entity","entity_kind","concordant_direction","n_conditions","n_papers",
             "n_papers_other_condition","other_conditions","best_tier"]]

45 headline-eligible convergence records:


## Recap — what these tables can and cannot support

| Table | Supports | Does **not** support |
|---|---|---|
| Associations | "these are co-reported more than chance" | "these interact" |
| Under-studied | "this is a candidate worth replicating" | "this is true but ignored" |
| Contradictions | "the literature disagrees here" | "one side is wrong" (check `broad_condition_flag`, `has_visible_moderator`) |
| Nulls | "this failed in these trials" | "this is ineffective" |
| Convergence | "co-reported direction across condition labels" | "shared mechanism" — non-POTS support is incidental mention in Tier 1 |